# 07-02 · 차입 계정이 없던 249곳의 재무상태표 이름 확인

| 항목 | 내용 |
|---|---|
| 하는 일 | 사전기간에 차입 3계정(단기차입금 등)이 하나도 안 잡힌 249곳의 재무상태표에 '차입·사채'처럼 보이는 이름이 있는지 **이름만** 확인하고, 전체 1,321곳에서도 같은 이름이 얼마나 되는지 비교 |
| 읽는 파일 | `outputs/05_계정확보/계정확보_기업별.csv`, `outputs/04_본수집/본수집_로그.csv`, 저장된 응답 |
| 만드는 파일 | `outputs/05_계정확보/차입계정_이름확인_249곳_기업별.csv`, `차입계정_이름빈도_249곳.csv`, `차입계정_이름빈도_전체_현재규칙밖.csv` |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 16초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

탐색용 단어 목록(차입·사채·금융부채·대출·리스부채·금융리스·단기금융·어음)은 **에이전트가 정한 탐색용이며 공식 분류가 아닙니다.** 이 결과로 E의 정의를 곧바로 정하지 않았습니다. E의 규칙은 `docs/A_결정_제안_근거표_11-05.md`에서 따로 제안한 것입니다.

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""차입 3계정이 사전기간에 하나도 없던 기업(249곳)의 재무상태표에 '차입·사채'처럼 보이는 이름의 계정이 있는지 이름만 확인한다.

이 스크립트가 하는 일 (금액은 읽지 않는다. 값 칸이 비어 있지 않은지만 본다. ICR·E는 계산하지 않는다)
  A. 249곳의 사전기간 보고서(상태 000)마다 재무상태표(BS)에서 차입 관련 단어가 들어간 계정 이름을 모은다.
  B. 비교용: 전체 1,321곳에서 기업마다 사전기간 보고서 1개(2020 사업보고서, 없으면 사전기간 중 가장 늦은 것)를 골라
     같은 단어가 들어간 계정 이름의 기업 수를 센다. 현재 규칙(단기차입금 등 3계정)에 안 잡히는 이름이 무엇인지 보려는 것이다.
  차입 관련 단어: 차입, 사채, 금융부채, 대출, 리스부채, 금융리스, 단기금융, 어음 (이 단어 목록은 에이전트가 정한 탐색용이며
  공식 분류가 아니다. 이 결과로 곧바로 E의 정의를 정하지 않는다.)

실행: python -X utf8 02_차입계정_이름확인.py  (저장된 파일만 읽음, 키 필요 없음)
"""
import csv
import json
import re
import sys
from collections import Counter, defaultdict
from pathlib import Path
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)
sys.path.insert(0, str(HERE.parent / "01_파일럿"))
import config  # noqa: E402
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
OUTDIR = config.BASE / "outputs" / "05_계정확보"
LOG = config.BASE / "outputs" / "04_본수집" / "본수집_로그.csv"
BYFIRM = OUTDIR / "계정확보_기업별.csv"
FS_DIR = config.RAW / "fs"
REP_ORDER = {"Q1": 0, "H1": 1, "Q3": 2, "FY": 3}
WORDS = re.compile(r"차입|사채|금융부채|대출|리스부채|금융리스|단기금융|어음")
CURRENT_RULE = re.compile(r"^(단기차입금|유동성장기차입금|유동성장기부채|유동성차입금|유동성사채|유동성장기사채)")

#### `nonempty()`

In [3]:
def nonempty(v):
    return v is not None and str(v).strip() not in ("", "-")

#### `is_pre()`

In [4]:
def is_pre(year, rep):
    return (2019, 0) <= (year, REP_ORDER[rep]) <= (2021, 1)

#### `bs_names()`

In [5]:
def bs_names(corp, year, rep, fs):
    p = FS_DIR / f"{corp}_{year}_{config.REPORTS[rep]}_{fs}.json"
    d = json.loads(p.read_text(encoding="utf-8"))
    out = set()
    for r in d.get("list", []):
        if r.get("sj_div") != "BS" or not nonempty(r.get("thstrm_amount")):
            continue
        nm = re.sub(r"\s+", "", r.get("account_nm", ""))
        if WORDS.search(nm):
            out.add(nm)
    return out

#### `main()`

이 함수(64줄) 안의 단계 — 코드 주석에서 뽑은 목차:
1. A. 249곳 전체 사전기간 보고서
2. B. 전체 비교 (기업당 보고서 1개)

In [6]:
def main():
    firms = list(csv.DictReader(open(BYFIRM, encoding="utf-8-sig")))
    none249 = {r["corp_code"] for r in firms if r["st_borrow_사전"] == "0" and r["cur_ltd_사전"] == "0" and r["cur_bond_사전"] == "0"}
    chosen = {}
    for r in csv.DictReader(open(LOG, encoding="utf-8-sig")):
        if r["status"] != "000":
            continue
        k = (r["corp_code"], int(r["year"]), r["report"])
        if k not in chosen or r["fs_div"] == "CFS":
            chosen[k] = r["fs_div"]

    # A. 249곳 전체 사전기간 보고서
    names_by_firm = defaultdict(set)
    n_reports = Counter()
    for (corp, year, rep), fs in chosen.items():
        if corp in none249 and is_pre(year, rep):
            n_reports[corp] += 1
            names_by_firm[corp] |= bs_names(corp, year, rep, fs)
    with_any = [c for c in none249 if names_by_firm[c]]
    print(f"A. 차입 3계정이 없는 {len(none249)}곳 중 사전기간 재무상태표에 차입 관련 단어가 든 계정이 있는 기업: {len(with_any)}곳, 하나도 없는 기업: {len(none249) - len(with_any)}곳")
    cnt = Counter(nm for c in none249 for nm in names_by_firm[c])
    print("  이름별 기업 수 (상위 25):")
    for nm, n in cnt.most_common(25):
        print(f"    {nm:30s} {n}곳")

    # B. 전체 비교 (기업당 보고서 1개)
    pick = {}
    for (corp, year, rep), fs in chosen.items():
        if not is_pre(year, rep):
            continue
        key = (year == 2020 and rep == "FY", year, REP_ORDER[rep])
        if corp not in pick or key > pick[corp][0]:
            pick[corp] = (key, year, rep, fs)
    allcnt = Counter()
    uncaught = Counter()
    no_word = 0
    for corp, (_, year, rep, fs) in pick.items():
        names = bs_names(corp, year, rep, fs)
        if not names:
            no_word += 1
        for nm in names:
            allcnt[nm] += 1
            if not CURRENT_RULE.search(nm):
                uncaught[nm] += 1
    print(f"\nB. 전체 {len(pick):,}곳(기업당 보고서 1개): 차입 관련 단어가 든 계정이 하나도 없는 기업 {no_word:,}곳")
    print("  현재 규칙(단기차입금·유동성장기차입금·유동성사채 계열)에 잡히지 않는 이름 (상위 30):")
    for nm, n in uncaught.most_common(30):
        print(f"    {nm:30s} {n}곳")

    OUTDIR.mkdir(parents=True, exist_ok=True)
    with open(OUTDIR / "차입계정_이름빈도_249곳.csv", "w", encoding="utf-8-sig", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["계정이름(공백제거)", "249곳 중 기업수"])
        w.writerows(cnt.most_common())
    with open(OUTDIR / "차입계정_이름빈도_전체_현재규칙밖.csv", "w", encoding="utf-8-sig", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["계정이름(공백제거)", "전체 기업수(기업당 보고서 1개)"])
        w.writerows(uncaught.most_common())
    with open(OUTDIR / "차입계정_이름확인_249곳_기업별.csv", "w", encoding="utf-8-sig", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["corp_code", "사전기간_보고서수", "차입관련_이름_있음", "이름들"])
        for c in sorted(none249):
            w.writerow([c, n_reports[c], bool(names_by_firm[c]), " | ".join(sorted(names_by_firm[c]))])
    print("\n저장:", OUTDIR)

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [7]:
main()

A. 차입 3계정이 없는 249곳 중 사전기간 재무상태표에 차입 관련 단어가 든 계정이 있는 기업: 242곳, 하나도 없는 기업: 7곳
  이름별 기업 수 (상위 25):
    단기금융상품                         93곳
    기타비유동금융부채                      86곳
    기타유동금융부채                       84곳
    유동리스부채                         78곳
    비유동리스부채                        74곳
    차입금                            69곳
    리스부채                           66곳
    기타금융부채                         65곳
    장기차입금                          39곳
    단기금융자산                         28곳
    전환사채                           26곳
    단기금융부채                         21곳
    장기금융부채                         19곳
    비유동금융부채                        17곳
    장기리스부채                         16곳
    단기차입부채                         13곳
    장기차입부채                         11곳
    유동성리스부채                        11곳
    유동차입금                          11곳
    유동금융부채                         10곳
    리스부채(비유동)                      10곳
    파생금융부채                         10곳
    차입금및사채                         10곳
    리스부


B. 전체 1,321곳(기업당 보고서 1개): 차입 관련 단어가 든 계정이 하나도 없는 기업 14곳
  현재 규칙(단기차입금·유동성장기차입금·유동성사채 계열)에 잡히지 않는 이름 (상위 30):
    장기차입금                          766곳
    단기금융상품                         522곳
    비유동리스부채                        449곳
    유동리스부채                         428곳
    기타유동금융부채                       406곳
    기타비유동금융부채                      397곳
    리스부채                           240곳
    전환사채                           185곳
    기타금융부채                         179곳
    유동성전환사채                        159곳
    유동성리스부채                        131곳
    사채                             130곳
    차입금                            97곳
    단기금융자산                         78곳
    장기리스부채                         48곳
    유동성금융리스부채                      31곳
    파생금융부채                         30곳
    단기차입부채                         26곳
    신주인수권부사채                       26곳
    단기리스부채                         26곳
    비유동금융리스부채                      25곳
    장기금융부채                         24곳
    리스부채(유동)        

### 저장된 결과 미리보기 (맨 위 5행)

위 실행이 만들었거나 이미 저장돼 있던 결과 파일을 표로 엽니다. 파일 전체는 엑셀로 여세요.

In [8]:
import pandas as pd
from IPython.display import display

_p = HERE.parents[1] / "outputs/05_계정확보/차입계정_이름빈도_전체_현재규칙밖.csv"
print(_p.name, "|", end=" ")
_df = pd.read_csv(_p, encoding="utf-8-sig", low_memory=False)
print(f"{len(_df):,}행 × {_df.shape[1]}열")
display(_df.head(5))

차입계정_이름빈도_전체_현재규칙밖.csv | 217행 × 2열


,계정이름(공백제거),전체 기업수(기업당 보고서 1개)
0,장기차입금,766
1,단기금융상품,522
2,비유동리스부채,449
3,유동리스부채,428
4,기타유동금융부채,406


### 알아둘 점 — 같은 빈도끼리의 줄 순서

`차입계정_이름빈도_*.csv`는 같은 기업 수(빈도)를 가진 이름들의 **줄 순서가 실행할 때마다 달라질 수 있습니다**(기업 코드 집합을 훑는 순서가 매번 달라서. 원본 스크립트에도 있던 성질이며 변환과 무관). 이름과 빈도 값은 항상 같습니다. 저장된 결과 파일은 처음 실행했을 때의 것을 그대로 두었습니다.